In [ ]:
# # Keles et al. (2024) from dandiset 000623: units, behaviour, gaze and single-neuron selectivity
#
# Paper: Keles, Dubois, Le, Tyszka, Kahn, Reed, Chung, Mamelak, Adolphs and Rutishauser (2024),
# "Multimodal single-neuron, intracranial EEG, and fMRI brain responses during movie watching in
# human patients", Scientific Data 11, 214, doi:10.1038/s41597-024-03029-1.
#
# Dataset: DANDI:000623, version 0.240130.2108 (doi:10.48324/dandi.000623/0.240130.2108). This is
# the version the paper cites (reference 54). The three published versions (0.240121.2058,
# 0.240130.2108, 0.240227.2023) list the same 29 assets with the same asset identifiers and sizes,
# so the choice does not change the data; the check is printed below.
#
# Claims file: `claims/000623/10.1038-s41597-024-03029-1.yaml`, written from the version of record
# (Europe PMC full text) and the paper's code, github.com/rutishauserlab/bmovie-release-NWB-BIDS at
# commit 8c35290, which the criteria follow and the helper `bmovie.py` ports.
#
# This notebook tests the claims on the units, behaviour, gaze and single neurons. The claims on
# HFB power of the microwire LFP and macro iEEG, the example units and channels of Figs. 6 and 7,
# and the comparison of modalities are in `hfb_keles_2024.ipynb`.
#
# | Claim | Here |
# |---|---|
# | cohort-emu-sessions, table1-demographics, neurons-total-and-areas, units-per-wire | tested |
# | firing-rate, isi-violations, waveform-snr, projection-distance, isolation-distance | tested |
# | gaze-missing-p53cs, gaze-similarity | tested |
# | recognition-auc, auc-run1-vs-run2, confidence-accuracy | tested |
# | memory-sessions-included, memory-cells-significant, memory-cells-not-significant | tested |
# | scenecut-labels (link), event-cells | tested, with the code's scene-cut annotations |
# | scanner-questions | tested on the code repository's copy of the in-scanner answers |
# | memory-lfp, memory-ieeg, event-lfp, event-ieeg, fig6-examples, fig7-examples, memory-modalities-differ, nwb-contents | in `hfb_keles_2024.ipynb` |
# | fmri-motion, fmri-tsnr, fmri-isc | blocked: the fMRI data on OpenNeuro hold no fMRIPrep derivatives, and fMRIPrep cannot be run here |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import json
import urllib.request
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from scipy import stats

import bmovie as bm

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

assets = bm.asset_urls()
with ProcessPoolExecutor(6) as ex:
    S = list(ex.map(bm.load_session, [u for _, u, _ in assets]))
cuts = bm.scenecuts()
ids = [s["identifier"] for s in S]
print(len(assets), "NWB assets;", round(sum(a[2] for a in assets) / 1e9, 2), "GB")

# the published versions hold the same assets
vers = {}
for v in ["0.240121.2058", "0.240130.2108", "0.240227.2023"]:
    with urllib.request.urlopen(f"https://api.dandiarchive.org/api/dandisets/000623/versions/{v}/assets/?page_size=100") as r:
        vers[v] = {a["path"]: (a["asset_id"], a["size"]) for a in json.load(r)["results"]}
for v, d in vers.items():
    print(v, len(d), "assets; identical to the pinned version:", d == vers[bm.DANDISET_VERSION])

In [ ]:
U_all = pd.concat([bm.units_U(s) for s in S], ignore_index=True)
rows = []
for s in S:
    t0, t1 = bm.movie_window(s)
    r = bm.recog_trials(s)
    u = U_all[U_all.session == s["identifier"]]
    rows.append(dict(session=s["identifier"], subject=s["subject"]["subject_id"],
                     movie_start=t0, movie_stop=round(t1, 3), recog_trials=len(r),
                     novel=int((r.label == "new").sum()), familiar=int((r.label == "old").sum()),
                     units=len(u), units_U=int(u.in_U.sum()),
                     micro=s["lfp"].get("LFP_micro", {}).get("shape"),
                     macro=s["lfp"].get("LFP_macro", {}).get("shape"),
                     lfp_start=[round(v["starting_time"], 4) for v in s["lfp"].values()],
                     lfp_rate=sorted({float(v["rate"]) for v in s["lfp"].values()}),
                     gaze_rate=s["gaze_rate"], frames=len(s["frame_times"])))
sess = pd.DataFrame(rows)
print(sess.to_string())
print("\nsubjects:", sess.subject.nunique(), " sessions:", len(sess), " units:", len(U_all),
      " units with a spike in the movie:", int(U_all.in_U.sum()))

print("\nTrials table columns:", list(S[0]["trials"].columns))
print("Units table columns:", S[0]["units_columns"])
print("Electrodes table columns:", list(S[0]["electrodes"].columns))
print("Behavior module:", S[0]["behavior_modules"])
print("Recognition responses (all sessions):",
      pd.Series(np.concatenate([bm.recog_trials(s).actual_response for s in S])).value_counts().sort_index().to_dict())
print("Response codes agree with response_correct in every session:",
      all((bm.recog_trials(s).correct_code == (bm.recog_trials(s).response_correct == 1)).all() for s in S))
print("Movie frames: data 0 to", int(max(s["frame_data"].max() for s in S)), "; first frame time",
      sorted({round(s["frame_times"][0], 3) for s in S}), "; frame interval (s) median",
      round(float(np.median(np.concatenate([np.diff(s["frame_times"]) for s in S]))), 4),
      "range", round(float(min(np.diff(s["frame_times"]).min() for s in S)), 4), "to",
      round(float(max(np.diff(s["frame_times"]).max() for s in S)), 4))
print("Movie duration (s): ", round(sess.movie_stop.min(), 2), "to", round(sess.movie_stop.max(), 2),
      "(11971 frames at 25 Hz is", 11971 / 25, "s)")
print("Gaze unit:", sorted({s["gaze_unit"] for s in S}), "; gaze comments:", sorted({s["gaze_comments"] for s in S}))
print("LFP unit:", sorted({v["unit"] for s in S for v in s["lfp"].values()}))
q = U_all
for c in ["rate", "isibelow", "peakSNR", "meanSNR", "isolationdist"]:
    print(f"{c:14s} min {np.nanmin(q[c]):.4g}  median {np.nanmedian(q[c]):.4g}  max {np.nanmax(q[c]):.4g}  NaN {int(q[c].isna().sum())}")
print("isibelow is described in the file as:", "'The proportion of inter-spike intervals (ISIs) which are shorter than 3 ms'")
print("Units whose electrode location has no hemisphere prefix:",
      int((~U_all.area.isin(bm.AREAS)).sum()), "; electrodes table index is 0..n-1 in every file:",
      all((s["electrodes"].index == np.arange(len(s["electrodes"]))).all() for s in S))

# duplicates: identical spike trains, gaze traces or trial tables across files
def sig(a):
    return hash(np.asarray(a).tobytes())
spk = {s["identifier"]: sig(np.concatenate([np.asarray(x) for x in s["units"].spike_times])) for s in S}
gz = {s["identifier"]: sig(s["gaze"][:5000]) for s in S}
tri = {s["identifier"]: sig(bm.recog_trials(s).start_time.to_numpy()) for s in S}
for name, d in [("spike times", spk), ("gaze", gz), ("recognition onsets", tri)]:
    print(f"distinct {name}: {len(set(d.values()))} of {len(d)}")

Checks of the description against the data, and of the clocks.

The LFP series state a starting time near 0 s, but the Usage Notes and the code
(`prep_filterLFP.py`, `LFP_PAD_us = 10`) say the LFP begins 10 s before the movie, which starts
at 0 s on the trials' clock. The LFP is aligned here as the code aligns it (file time minus
10 s); `hfb_keles_2024.ipynb` tests that alignment against the file's own starting time with
the scene-cut responses.

The spikes share the clock of the trials table if units respond to recognition-trial onsets
at the stated times and less at shifted times. The next cell counts, for each shift, the units
whose spike count in 0.2 to 1.7 s after onset differs from the 1.5 s before onset (Wilcoxon
signed-rank over the 40 trials, p < 0.01).

In [ ]:
rng_clock = np.random.default_rng(1)
shifts = [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
res = {}
for sh in shifts:
    n_sig = n_tot = 0
    for s in S:
        ons = bm.recog_trials(s).start_time.to_numpy() + sh
        for st, inU in zip(s["units"].spike_times, bm.units_U(s).in_U):
            if not inU:
                continue
            post = bm.window_counts(np.asarray(st), ons, (0.2, 1.7))
            pre = bm.window_counts(np.asarray(st), ons, (-1.5, -0.0001))
            n_tot += 1
            if np.any(post != pre) and stats.wilcoxon(post, pre).pvalue < 0.01:
                n_sig += 1
    res[sh] = (n_sig, n_tot)
for sh, (a, b) in res.items():
    print(f"onsets shifted by {sh:+.0f} s: {a} of {b} units respond ({100 * a / b:.1f}%)")

## cohort-emu-sessions

> The electrophysiology data come from 16 participants and 29 EMU sessions (runs); 13
> participants performed two runs and P44CS, P58CS and P60CS one, as Table 1 lists.

In [ ]:
table1_runs = {"P41CS": 2, "P42CS": 2, "P43CS": 2, "P44CS": 1, "P47CS": 2, "P48CS": 2, "P49CS": 2,
               "P51CS": 2, "P53CS": 2, "P54CS": 2, "P55CS": 2, "P56CS": 2, "P57CS": 2, "P58CS": 1,
               "P60CS": 1, "P62CS": 2}   # Table 1, '# of EMU runs'
runs = pd.Series([i.split("_")[0] for i in ids]).value_counts().sort_index()
print("sessions:", len(ids), " participants:", runs.size, " with two runs:", int((runs == 2).sum()))
print("subject_id (CSnn) agrees with the identifier (PnnCS) in every file:",
      all(s["subject"]["subject_id"] == "CS" + s["identifier"][1:3] for s in S))
cmp = pd.DataFrame({"dataset": runs, "table1": pd.Series(table1_runs)})
print(cmp.T.to_string())
print("every participant's run count matches Table 1:", bool((cmp.dataset == cmp.table1).all()))

**Result: reproduced.** 29 sessions from 16 participants, 13 with two runs; every participant's
run count matches Table 1.

## table1-demographics

> Of the 20 participants 11 are female, and their mean age is 37.75 years (SD 13.86); the age,
> sex and epilepsy diagnosis of each participant are as in Table 1.

The four fMRI-only participants are not in the dandiset; their values are those of Table 1, as
`gen_table1_subj_info.py` enters them by hand (an outside input, `with_inputs`).

In [ ]:
table1 = pd.DataFrame([
    ("P41CS", 21, "F", "Left Other"), ("P42CS", 25, "F", "Not Localized"),
    ("P43CS", 42, "F", "Left Mesial Temporal"), ("P44CS", 53, "F", "Right Mesial Temporal"),
    ("P45CS", 29, "F", "Bitemporal"), ("P46CS", 41, "M", "NA"),
    ("P47CS", 32, "M", "Right Mesial Temporal"), ("P48CS", 32, "F", "Left Mesial Temporal"),
    ("P49CS", 24, "F", "Left Mesial Temporal"), ("P50CS", 25, "M", "Right Temporal Neocortical"),
    ("P51CS", 17, "M", "Not Localized"), ("P53CS", 60, "M", "Bilateral Independent Temporal"),
    ("P54CS", 59, "F", "Right Mesial Temporal"), ("P55CS", 43, "F", "Right Mesial Temporal"),
    ("P56CS", 48, "M", "Bilateral Independent Temporal"), ("P57CS", 46, "M", "Left Other"),
    ("P58CS", 32, "F", "Right Lateral Frontal"), ("P59CS", 34, "M", "Left Mesial Temporal"),
    ("P60CS", 67, "M", "Left Mesial Temporal"), ("P62CS", 25, "F", "Right Mesial Temporal"),
], columns=["ID", "age", "sex", "diagnosis"]).set_index("ID")   # Table 1 of the paper
fmri_only = ["P45CS", "P46CS", "P50CS", "P59CS"]
emu = {}
for s in S:
    pid = s["identifier"].split("_")[0]
    emu.setdefault(pid, set()).add((int(s["subject"]["age"].removeprefix("P").removesuffix("Y")),
                                    s["subject"]["sex"],
                                    s["subject"]["description"].removeprefix("epilepsy diagnosis: ")))
print("participants whose two runs disagree on age, sex or diagnosis:", [k for k, v in emu.items() if len(v) > 1])
emu_df = pd.DataFrame([(k, *list(v)[0]) for k, v in emu.items()], columns=["ID", "age", "sex", "diagnosis"]).set_index("ID")
chk = emu_df.join(table1.add_suffix("_table1"))
chk["match"] = (chk.age == chk.age_table1) & (chk.sex == chk.sex_table1) & (chk.diagnosis == chk.diagnosis_table1)
print(chk.to_string())
all20 = pd.concat([emu_df, table1.loc[fmri_only]])
print("\nparticipants:", len(all20), " female:", int((all20.sex == "F").sum()))
print("mean age %.2f, SD (ddof 0) %.2f, SD (ddof 1) %.2f" % (all20.age.mean(), all20.age.std(ddof=0), all20.age.std(ddof=1)))
print("EMU participants only (16): mean %.2f, SD %.2f" % (emu_df.age.mean(), emu_df.age.std(ddof=0)))

**Result: reproduced.** The 16 EMU participants' age, sex and diagnosis in the files match Table
1, and with the four fMRI-only participants from Table 1 the 20 have 11 women and a mean age of
37.75 years (SD 13.86, ddof 0).

## neurons-total-and-areas

> The dataset holds 1450 single units from amygdala (479), hippocampus (261), ACC (236), preSMA
> (299) and vmPFC (175).

In [ ]:
U = U_all[U_all.in_U]
print("units in the files:", len(U_all), " units of U (a spike in the movie):", len(U))
print(pd.DataFrame({"U": U.area.value_counts(), "all units": U_all.area.value_counts()}).T.to_string())
print("units without a spike in the movie:", U_all[~U_all.in_U][["session", "unit_id_session", "area"]].to_string())
print("area lookup by .iloc and by .loc agree for every unit:", bool((U_all.area == U_all.area_loc).all()))
print("\nper session:")
print(U.groupby(["session", "area"]).size().unstack(fill_value=0).to_string())

**Result: reproduced.** 1450 units of U: amygdala 479, hippocampus 261, ACC 236, preSMA 299
and vmPFC 175, each as stated. The files hold 1457 units; 7 have no spike in the movie.

## units-per-wire

> There were on average 1.99 units per wire, counting only wires with at least one unit.

In [ ]:
per_wire = U_all.groupby(["session", "electrode_id"]).size()
print("session-wire pairs with a unit:", len(per_wire), " units:", int(per_wire.sum()))
print("mean units per wire: %.3f" % per_wire.mean(), " distribution:", per_wire.value_counts().sort_index().to_dict())
print("units of U only: %.3f" % U.groupby(["session", "electrode_id"]).size().mean())

**Result: reproduced.** 1.990 units per wire with at least one unit, over 732 session and wire
pairs and all 1457 units (1.981 with the 1450 units of U).

## firing-rate

> The mean firing rate of the neurons was 2.47 +- 3.32 spikes/s (mean +- s.d.).

In [ ]:
print("units:", len(U), " mean %.3f  s.d. (ddof 0) %.3f  s.d. (ddof 1) %.3f  range %.3f to %.2f spikes/s"
      % (U.rate.mean(), U.rate.std(ddof=0), U.rate.std(ddof=1), U.rate.min(), U.rate.max()))
print("by participant:")
print(U.assign(p=U.session.str[:5]).groupby("p").rate.agg(["count", "mean", "std"]).round(2).T.to_string())

**Result: reproduced.** Mean 2.471 spikes/s, s.d. 3.326 (ddof 0; 3.327 with ddof 1) over the
1450 units of U.

## isi-violations

> The percentage of interspike intervals below 3 ms was 0.40% +- 0.53%.

In [ ]:
a = U.isibelow
b = a[a < 3]
print("(a) all units of U: n %d, mean %.3f, s.d. %.3f (ddof 1 %.3f)" % (len(a), a.mean(), a.std(ddof=0), a.std(ddof=1)))
print("(b) units with isibelow < 3: n %d, mean %.3f, s.d. %.3f (ddof 1 %.3f)" % (len(b), b.mean(), b.std(ddof=0), b.std(ddof=1)))
print("range of isibelow: %.3f to %.3f; units at or above 3: %d" % (a.min(), a.max(), int((a >= 3).sum())))

**Result: reproduced.** Under reading (b), the 1426 units with less than 3% plotted in Fig. 2c,
the mean is 0.404% and the s.d. 0.529%, both within tolerance. Under reading (a), all 1450 units,
0.462% and 0.727%, both outside. The column is described in the files as a proportion, but its
values (up to 10.35) are percentages, as the paper reports them.

## waveform-snr

> The peak SNR of the mean waveform was 5.95 +- 3.48 and the mean SNR of the whole waveform 2.07
> +- 1.13.

In [ ]:
for c in ["peakSNR", "meanSNR"]:
    print("%s: n %d, mean %.3f, s.d. %.3f (ddof 1 %.3f), range %.2f to %.2f"
          % (c, U[c].notna().sum(), U[c].mean(), U[c].std(ddof=0), U[c].std(ddof=1), U[c].min(), U[c].max()))

**Result: reproduced.** Peak SNR 5.948 +- 3.477 and mean SNR 2.066 +- 1.128 over the 1450 units
(ddof 0; ddof 1 changes the third decimal).

## projection-distance

> The pairwise projection distance between neurons isolated on the same wire was 11.98 +- 8.93
> s.d. units.

In [ ]:
pairs = []
n_entries = 0
for s in S:
    for v in s["electrodes"]["pairwise_distances"]:
        if v != "NA" and v != "":
            n_entries += 1
            pairs.extend(map(float, v.split("_")))
pairs = np.array(pairs)
print("wires with an entry:", n_entries, " pairs:", len(pairs))
print("mean %.3f, s.d. %.3f (ddof 1 %.3f), range %.2f to %.2f" % (pairs.mean(), pairs.std(), pairs.std(ddof=1), pairs.min(), pairs.max()))
exp_pairs = int(sum(k * (k - 1) / 2 for k in per_wire))
print("pairs expected from units per wire (k(k-1)/2):", exp_pairs)

**Result: reproduced.** 11.982 +- 8.933 over 1242 pairs from 403 wires. The units per wire would
give 1266 pairs, so 24 pairs of units on the same wire have no entry.

## isolation-distance

> The median isolation distance of the units was 24.90.

In [ ]:
iso = U.isolationdist.to_numpy()
n_nan = int(np.isnan(iso).sum())
iso = iso[~np.isnan(iso)]
n_big = int((iso >= 1e18).sum())
iso = iso[iso < 1e18]
print("units:", len(U), " NaN removed:", n_nan, " >= 1e18 removed:", n_big, " kept:", len(iso))
print("median %.3f; range %.3g to %.3g" % (np.median(iso), iso.min(), iso.max()))

**Result: reproduced.** Median 24.904 over 1001 units; 448 units of U have no isolation distance
(NaN) and one has 3.4e27, removed as the code does.

## gaze-missing-p53cs

> Gaze data were missing for more than half of the movie only for P53CS (74% in run 1, 84% in
> run 2), whose gaze was excluded from the eye-tracking analysis.

In [ ]:
G = {}
for s in S:
    bins, miss, ppd = bm.gaze_bins(s)
    G[s["identifier"]] = dict(bins=bins, missing=miss, ppd=ppd,
                             n=sum(len(b) for b in bins))
gm = pd.Series({k: 100 * v["missing"] for k, v in G.items()}).sort_index()
print(gm.round(1).to_string())
print("sessions at or above 50%:", list(gm[gm >= 50].index))
print("sessions with gaze stored by timestamps:", [s["identifier"] for s in S if s["gaze_rate"] is None])

**Result: reproduced.** P53CS has 73.6% of gaze missing in run 1 and 83.6% in run 2, and its two
runs are the only sessions at or above 50%.

## gaze-similarity

> Each participant's gaze was highly correlated with the reference heatmap of the other
> participants (Pearson's r = 0.75 +- 0.12) and little gaze was missing (7% +- 8%).

In [ ]:
keep = sorted(k for k in G if G[k]["missing"] < 0.5)
sigma = np.mean([G[k]["ppd"] for k in sorted(G)])   # mean over sessions, before exclusion (code)
print("sessions kept:", len(keep), " sigma (pixels per degree): %.3f" % sigma)
subj = np.array([k.split("_")[0] for k in keep])
corr = np.full((len(keep), bm.NBINS), np.nan)
for t in range(bm.NBINS):
    for p in np.unique(subj):
        others = np.vstack([G[k]["bins"][t] for k, sj in zip(keep, subj) if sj != p])
        ref = bm.heatmap(others, sigma)
        for i in np.where(subj == p)[0]:
            h = bm.heatmap(G[keep[i]]["bins"][t], sigma, nan_ratio=0.5)
            if h is not None and ref is not None:
                corr[i, t] = np.corrcoef(ref.ravel(), h.ravel())[0, 1]
z = np.nanmean(np.arctanh(corr), 1)
print("per session r:", dict(zip(keep, np.tanh(z).round(3))))
print("bins used per session: min %d, max %d of %d" % (np.isfinite(corr).sum(1).min(), np.isfinite(corr).sum(1).max(), bm.NBINS))
print("mean r = tanh(mean z) = %.3f; s.d. = tanh(s.d. z, ddof 0) = %.3f; ddof 1 %.3f"
      % (np.tanh(z.mean()), np.tanh(z.std()), np.tanh(z.std(ddof=1))))
print("per participant (mean z over runs): mean %.3f, s.d. %.3f" % (
    np.tanh(pd.Series(z, index=subj).groupby(level=0).mean().mean()),
    np.tanh(pd.Series(z, index=subj).groupby(level=0).mean().std(ddof=0))))
mk = gm[keep]
print("missing gaze (a) the %d kept sessions: mean %.2f%%, s.d. %.2f%% (ddof 1 %.2f)" % (len(mk), mk.mean(), mk.std(ddof=0), mk.std(ddof=1)))
print("missing gaze (b) all %d sessions: mean %.2f%%, s.d. %.2f%% (ddof 1 %.2f)" % (len(gm), gm.mean(), gm.std(ddof=0), gm.std(ddof=1)))
# a check of the gaze clock: the same correlation with each session's gaze shifted by 5 s
corr_sh = []
for i, k in enumerate(keep[:6]):
    vals = []
    for t in range(5, bm.NBINS):
        others = np.vstack([G[kk]["bins"][t] for kk, sj in zip(keep, subj) if sj != subj[i]])
        h = bm.heatmap(G[k]["bins"][t - 5], sigma, nan_ratio=0.5)
        if h is not None:
            vals.append(np.corrcoef(bm.heatmap(others, sigma).ravel(), h.ravel())[0, 1])
    corr_sh.append(np.tanh(np.nanmean(np.arctanh(vals))))
print("clock check, first 6 sessions: r with true timing", np.tanh(z[:6]).round(3), "; with gaze lagged 5 s", np.round(corr_sh, 3))

**Result: reproduced.** Over the 27 sessions left, r = 0.751 with s.d. 0.122 (inverse
Fisher-transformed, ddof 0). Missing gaze is 6.63% +- 8.40% under reading (a), the 27 sessions,
within tolerance; under reading (b), all 29 sessions, 11.59% +- 20.01%, outside. Lagging a
session's gaze by 5 s lowers its correlation from 0.737 to 0.834 to 0.290 to 0.317 (first
six sessions), consistent with the gaze sharing the movie's clock.

## recognition-auc

> Participants remembered the movie well; their recognition accuracy across confidence levels
> was 0.79 +- 0.19 AUC.

In [ ]:
beh = []
for s in S:
    r = bm.recog_trials(s)
    resp = r.actual_response.to_numpy()
    auc = bm.session_auc((r.label == "old").to_numpy(), resp)
    ok, n_new, n_old = bm.rule_R(s)
    beh.append(dict(session=s["identifier"], participant=s["identifier"][:5], run=int(s["identifier"][-1]),
                    trials=len(r), auc=auc, correct_new=n_new, correct_old=n_old, ruleR=ok))
beh = pd.DataFrame(beh).set_index("session")
print(beh.round(3).to_string())
print("\n(a) 29 sessions: mean %.3f, s.d. %.3f (ddof 1 %.3f)" % (beh.auc.mean(), beh.auc.std(ddof=0), beh.auc.std(ddof=1)))
pp = beh.groupby("participant").auc.mean()
print("(b) %d participants: mean %.3f, s.d. %.3f (ddof 1 %.3f)" % (len(pp), pp.mean(), pp.std(ddof=0), pp.std(ddof=1)))

**Result: reproduced.** Over the 29 sessions (reading a) the AUC is 0.789 +- 0.189 (ddof 0), both
within tolerance. Over the 16 participants (reading b) it is 0.787 +- 0.134, the s.d. outside
tolerance; reading (a) is the legend's "all sessions".

## auc-run1-vs-run2

> Memory accuracy did not differ between the first and second run for the 13 participants with
> two runs (0.79 +- 0.14 against 0.79 +- 0.24 AUC, p = 0.960, paired t-test).

In [ ]:
two = beh.groupby("participant").filter(lambda d: len(d) == 2)
r1 = two[two.run == 1].set_index("participant").auc
r2 = two[two.run == 2].set_index("participant").auc.loc[r1.index]
t = stats.ttest_rel(r1, r2)
print("participants:", len(r1))
print("run 1: mean %.3f, s.d. %.3f (ddof 1 %.3f)" % (r1.mean(), r1.std(ddof=0), r1.std(ddof=1)))
print("run 2: mean %.3f, s.d. %.3f (ddof 1 %.3f)" % (r2.mean(), r2.std(ddof=0), r2.std(ddof=1)))
print("paired t-test: t = %.3f, p = %.3f; mean difference (run 1 - run 2) %.3f" % (t.statistic, t.pvalue, (r1 - r2).mean()))

**Result: reproduced.** For the 13 participants with two runs, run 1 0.788 +- 0.138 and run 2
0.792 +- 0.245 AUC (ddof 0), all four values within tolerance; paired t-test p = 0.958 (mean
difference -0.004), so no difference is found.

## confidence-accuracy

> Recognition accuracy was significantly higher for high-confidence than for low-confidence
> judgments (p < 0.001, paired t-test).

In [ ]:
ca = []
for s in S:
    r = bm.recog_trials(s)
    resp = r.actual_response.to_numpy()
    auc = beh.loc[s["identifier"], "auc"]
    if bm.check_inclusion(resp, auc):
        mode, hi, lo = bm.confidence_accuracy((r.label == "old").to_numpy(), resp)
        ca.append(dict(session=s["identifier"], split=mode, high=hi, low=lo))
ca = pd.DataFrame(ca).set_index("session")
print(ca.round(1).to_string())
t = stats.ttest_rel(ca.high, ca.low)
print("\nsessions included:", len(ca), " high %.1f +- %.1f%%, low %.1f +- %.1f%%" % (ca.high.mean(), ca.high.std(ddof=0), ca.low.mean(), ca.low.std(ddof=0)))
print("paired t-test: t = %.3f, p = %.2g; high above low in %d of %d sessions" % (t.statistic, t.pvalue, int((ca.high > ca.low).sum()), len(ca)))

**Result: reproduced.** 22 sessions pass check_inclusion (all with split 1); accuracy is 89.9%
for high and 72.0% for low confidence, higher in 18 of 22 sessions; paired t-test p = 0.00027.

## memory-sessions-included

> Of the 1450 neurons, 1266 entered the memory analysis; the rest came from runs excluded for
> poor behavior (P44CS run 1, P51CS run 2, P56CS run 2, P57CS runs 1 and 2).

In [ ]:
listed = {"P44CS_R1", "P51CS_R2", "P56CS_R2", "P57CS_R1", "P57CS_R2"}
n_U = U.groupby("session").size()
readings = {
    "(a) rule R": set(beh.index[~beh.ruleR]),
    "(b) the five listed runs": listed,
    "(c) AUC <= 0.6 or rule R": set(beh.index[(beh.auc <= 0.6) | ~beh.ruleR]),
}
for name, rem in readings.items():
    print("%-26s removes %-60s units kept %d" % (name, sorted(rem), int(n_U.drop(sorted(rem), errors="ignore").sum())))
print("\nAUC and correct trials of the sessions any reading removes:")
print(beh.loc[sorted(set.union(*readings.values())), ["auc", "correct_new", "correct_old", "ruleR"]].round(3).to_string())
memory_sessions = sorted(set(beh.index) - readings["(a) rule R"])

**Result: reproduced.** All three readings remove the same five runs, the five the paper lists
(P44CS run 1, P51CS run 2, P56CS run 2, P57CS runs 1 and 2), and keep 1266 units. P44CS run 1 is
removed for having no correct novel trials although its AUC is 0.726.

## memory-cells-significant and memory-cells-not-significant

> Significant proportions of cells were memory selective in the amygdala (32/436, p = 0.015),
> ACC (22/186, p = 0.002) and vmPFC (14/150, p = 0.016).

> The proportions of memory-selective cells in the hippocampus (14/238, p = 0.30) and preSMA
> (17/256, p = 0.17) were not significant.

Units of U in the sessions kept by rule R (the reading of memory-sessions-included that gives
1266 units); spike counts in 0.2 to 1.7 s after each trial's start; correct trials; bootstrap B
(novel against familiar, balanced, seed 13); permutation test P with 2,000 shuffles (seed 0).

In [ ]:
blocks_m, table_m = bm.cell_blocks(S, "memory", sessions=set(memory_sessions))
cells_m, sum_m = bm.selection_and_null(blocks_m, table_m, balanced=True, positive_only=False, seed=0)
paper = pd.DataFrame({"area": ["amygdala", "hippocampus", "ACC", "preSMA", "vmPFC"],
                      "paper_selective": [32, 14, 22, 17, 14], "paper_n": [436, 238, 186, 256, 150],
                      "paper_p": [0.015, 0.30, 0.002, 0.17, 0.016]})
print(sum_m.merge(paper).round(4).to_string())
print("\nselective cells: %d of %d; novel > familiar in %d" % (cells_m.selective.sum(), len(cells_m), int((cells_m.selective & (cells_m["diff"] > 0)).sum())))
print("by participant (selective / cells):")
cp = cells_m.assign(p=cells_m.session.str[:5]).groupby("p").agg(sel=("selective", "sum"), n=("selective", "size"))
print(cp.T.to_string())

**Result: memory-cells-significant reproduced.** Amygdala 32/436 (permutation p = 0.0065), ACC
21/186 (p = 0.0005) and vmPFC 14/150 (p = 0.014): each below 0.05, with counts within tolerance
and the denominators exact.

**Result: memory-cells-not-significant reproduced.** Hippocampus 14/238 (p = 0.227) and preSMA
17/256 (p = 0.135), neither below 0.05, with the paper's counts.

## scenecut-labels

> The movie has 80 continuity cuts and 13 scene changes, and the annotation file's cuts can be
> placed in each session through the frame times in the NWB files.

In [ ]:
print("cuts:", len(cuts), cuts.kind.value_counts().to_dict())
print("shot_start_fr range:", int(cuts.shot_start_fr.min()), "to", int(cuts.shot_start_fr.max()),
      "; frames per session:", sorted({len(s["frame_times"]) for s in S}))
ok = []
for s in S:
    t0, t1 = bm.movie_window(s)
    o = bm.cut_onsets(s, cuts)
    ok.append(bool(np.all((o >= t0) & (o <= t1)) and cuts.shot_start_fr.max() <= len(s["frame_times"])))
print("sessions where every cut onset falls in the movie window:", sum(ok), "of", len(S))
dev = np.concatenate([bm.cut_onsets(s, cuts) - cuts.shot_start_t.to_numpy() for s in S])
print("cut onset minus the file's shot_start_t (s): median %.3f, range %.3f to %.3f" % (np.median(dev), dev.min(), dev.max()))

**Result: reproduced.** The annotation file has 80 continuity cuts and 13 scene changes by rule
S, and every cut onset falls in the movie window in all 29 sessions; the onsets from the frame
times are within 0 to 20 ms of the file's own shot start times.

## event-cells

> Significant proportions of cells were event selective (scene change against continuity cut) in
> the vmPFC (10/175, p = 0.005) and preSMA (20/299, p < 0.001), but not in the amygdala (15/479,
> p = 0.081), ACC (7/236, p = 0.224) or hippocampus (8/261, p = 0.199).

In [ ]:
blocks_e, table_e = bm.cell_blocks(S, "event", cuts=cuts)
cells_e, sum_e = bm.selection_and_null(blocks_e, table_e, balanced=False, positive_only=True, seed=1)
paper_e = pd.DataFrame({"area": ["amygdala", "hippocampus", "ACC", "preSMA", "vmPFC"],
                        "paper_selective": [15, 8, 7, 20, 10], "paper_n": [479, 261, 236, 299, 175],
                        "paper_p": [0.081, 0.199, 0.224, 0.001, 0.005]})
print(sum_e.merge(paper_e).round(4).to_string())
print("\nvariant, either direction (p < 0.05):",
      {a: int(cells_e.selective_any_direction[cells_e.area == a].sum()) for a in bm.AREAS})
print("cells selective with change < continuity:",
      {a: int((cells_e.selective_any_direction & (cells_e["diff"] < 0))[cells_e.area == a].sum()) for a in bm.AREAS})
cp = cells_e.assign(p=cells_e.session.str[:5]).groupby("p").agg(sel=("selective", "sum"), n=("selective", "size"))
print("by participant:"); print(cp.T.to_string())

**Result: reproduced.** The counts are the paper's in every area: vmPFC 10/175 (p = 0.004),
preSMA 20/299 (p = 0.0005), amygdala 15/479 (p = 0.089), ACC 7/236 (p = 0.226) and hippocampus
8/261 (p = 0.207). In either direction many more cells are selective (amygdala 72, preSMA 42), most
of them with weaker responses to scene changes; the paper's numbers are those of the code's
one-direction rule.

## scanner-questions

> In the scanner, participants answered on average 5.97 +- 0.98 of seven questions on the movie
> correctly (across participants and runs).

Not in the dandiset (`data: elsewhere`). The in-scanner answers are read from the code
repository's `assets/annotations/scanner_questionnaire.csv` at commit 8c35290, which the claims
file names as one of the two copies.

In [ ]:
url_q = ("https://raw.githubusercontent.com/rutishauserlab/bmovie-release-NWB-BIDS/"
         f"{bm.CODE_COMMIT}/assets/annotations/scanner_questionnaire.csv")
with urllib.request.urlopen(url_q) as r:
    qn = pd.read_csv(r)
print(qn.to_string())
vals = pd.concat([qn["R1 (corrects)"], qn["R2 (corrects)"]]).dropna()
print("\nparticipants:", len(qn), " runs:", len(vals))
print("mean %.3f, s.d. %.3f (ddof 1 %.3f)" % (vals.mean(), vals.std(ddof=0), vals.std(ddof=1)))
sums_ok = all((qn[[f"R{k}_Q{i}" for i in range(1, 8)]].sum(axis=1) == qn[f"R{k} (corrects)"]).all() for k in (1, 2))
print("per-question answers sum to the totals:", sums_ok)

**Result: reproduced.** Over 15 participants and 30 runs, 5.967 +- 0.983 correct answers (ddof
0), both within tolerance.

## Summary

| Claim | Outcome |
|---|---|
| cohort-emu-sessions | reproduced |
| table1-demographics | reproduced (with the Table 1 values of the four fMRI-only participants) |
| neurons-total-and-areas | reproduced |
| units-per-wire | reproduced |
| firing-rate | reproduced |
| isi-violations | reproduced, under reading (b), units below 3% |
| waveform-snr | reproduced |
| projection-distance | reproduced |
| isolation-distance | reproduced |
| gaze-missing-p53cs | reproduced |
| gaze-similarity | reproduced, missing gaze under reading (a), the 27 sessions kept |
| recognition-auc | reproduced, under reading (a), the 29 sessions |
| auc-run1-vs-run2 | reproduced |
| confidence-accuracy | reproduced |
| memory-sessions-included | reproduced (all readings agree) |
| memory-cells-significant | reproduced |
| memory-cells-not-significant | reproduced |
| scenecut-labels | reproduced |
| event-cells | reproduced |
| scanner-questions | reproduced (code repository's copy of the answers) |

The LFP, iEEG, example, modality and file-content claims are in `hfb_keles_2024.ipynb`; the fMRI
claims are blocked (see the header).

How the analysis differs from the paper's. The permutation tests are seeded and share each
shuffle's bootstrap resamples among the cells of a session (see `bmovie.py`); the paper's code
draws them per cell and unseeded. Selectivity is computed on spike counts, which have the sign
and ties of the code's rates. Where the claims file names readings, all are shown.

What was found about the dataset. The files match the paper closely: every count of sessions,
participants, units and cuts is exact, and the quality metrics, behaviour and selectivity counts
match to the reported precision. The units table describes `isibelow` as a proportion, but it
holds percentages. 448 of the 1450 units have no isolation distance and one has 3.4e27. The
electrodes table lists 1242 projection distances where the units per wire imply 1266 pairs.
Three sessions (P54CS run 2, P56CS runs 1 and 2) store gaze with timestamps, the others with a
rate. P53CS run 2 has 34 recognition trials, not 40. The fMRI-only participants' demographics
are not in the dandiset.